# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fraz-Rasool/ML-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**Contract answers**

1. **One row:** the raw warehouse table is one row per `report_date × client_hash_id × content_hash_id`. For my Lane 4 analysis, I aggregate those daily rows to **one row per client × content item for March 2026**.
2. **Tables:** `fact_content_daily_performance` is the main source. I use `dim_content` only when content-level metadata is needed; this week's five-feature frame is intentionally built from the daily fact alone.
3. **Time window:** March 1–31, 2026 is the feature / decision window. April 1–30, 2026 is a **future outcome window** used only to define the development proxy, not as an input feature.
4. **What I rank/predict:** I rank pages by a leakage-safe **future CTR under-capture proxy**: whether April CTR is below the April position-tier benchmark, after requiring enough April impressions for the rate to be interpretable.
5. **Deliberate exclusion:** I exclude `client_hash_id` and `content_hash_id` from model features because they are identifiers for joins/grouping, not meaningful signals. I also keep all April outcome fields out of March features.

**Decision moment:** March 31, 2026. Every feature below must be knowable by that date.


In [2]:
# Setup: Colab + gated Hugging Face warehouse
# Never paste the HF token into this notebook. Store it in Colab Secrets as HF_TOKEN.

import os
import duckdb
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN is missing. In Colab: add a Secret named HF_TOKEN, "
        "then Run all again. Do not paste the token into a cell."
    )

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf_secret (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance/**/*.parquet"

print("DuckDB connection ready.")
print("Source:", FACT)
print("Feature window: 2026-03-01 to 2026-03-31")
print("Outcome window: 2026-04-01 to 2026-04-30")


DuckDB connection ready.
Source: hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet
Feature window: 2026-03-01 to 2026-03-31
Outcome window: 2026-04-01 to 2026-04-30


## 2. Fields: feature / label / context / excluded

**Feature fields — safe at the March 31 decision moment**

- `march_impressions`: March GSC impressions; available because the full March window has closed.
- `march_clicks`: March GSC clicks; available at the same decision moment.
- `march_ctr`: March clicks ÷ March impressions × 100; computed only from March data.
- `march_avg_position`: impression-weighted March average position; computed only from March data.
- `march_impression_days`: number of March days with GSC impressions; available from the March daily fact.

**Label / proxy — never a feature**

- `future_under_capture`: April 2026 outcome proxy. It is 1 when April CTR is below the April position-tier benchmark, with an April impression floor. It is created from future April observations.

**Context**

- `client_hash_id`, `content_hash_id`: pseudonymous IDs used for grouping, joins, and grouped splits only.
- `report_date`: used to select the time windows, not as a model feature.

**Excluded**

- April clicks, impressions, CTR, position, and the April-derived label: future information unavailable at the March decision moment.
- `gsc_data_available`: used as a data-quality/availability filter, not as a predictive feature.
- Any product-generated recommendation/action field, if present: excluded because it can encode downstream decisions rather than independent evidence.

**Output:** a ranked/scored list of content items that deserve CTR/engagement review first.


In [3]:
# No new warehouse query here: this cell records the contract fields explicitly.
feature_contract = pd.DataFrame([
    ["march_impressions", "Feature", "Known when March closes because it is aggregated from March GSC daily rows."],
    ["march_clicks", "Feature", "Known when March closes because it is aggregated from March GSC daily rows."],
    ["march_ctr", "Feature", "Computed from March clicks and impressions before the April outcome occurs."],
    ["march_avg_position", "Feature", "Computed from March GSC position observations before the decision."],
    ["march_impression_days", "Feature", "Count of March days with measurable GSC impressions."],
    ["future_under_capture", "Label/proxy", "Uses April outcome data, so it is not available at the March decision moment."],
    ["client_hash_id / content_hash_id", "Context", "Needed for joins, grouping, and splits; identifiers are not predictive features."],
    ["April outcome fields", "Excluded", "Future information would leak the answer into the feature window."],
], columns=["field", "bucket", "available when / why"])
display(feature_contract)


,field,bucket,available when / why
0,march_impressions,Feature,Known when March closes because it is aggregat...
1,march_clicks,Feature,Known when March closes because it is aggregat...
2,march_ctr,Feature,Computed from March clicks and impressions bef...
3,march_avg_position,Feature,Computed from March GSC position observations ...
4,march_impression_days,Feature,Count of March days with measurable GSC impres...
5,future_under_capture,Label/proxy,"Uses April outcome data, so it is not availabl..."
6,client_hash_id / content_hash_id,Context,"Needed for joins, grouping, and splits; identi..."
7,April outcome fields,Excluded,Future information would leak the answer into ...


## 3. Verify it with queries (grain, counts, availability) + build five features + trap

The three verification queries below are deliberately small and correspond to the three required facts:

1. **Grain:** duplicate `(report_date, client_hash_id, content_hash_id)` keys should return zero rows.
2. **Slice count + date span:** March row count and minimum/maximum date.
3. **Availability:** use `IS TRUE` and report how many March rows survive.

After those checks, the same month is aggregated into the five-feature frame. The April label is kept separate so the feature frame remains decision-time safe.


In [4]:
# -----------------------------
# Verification query 1 — GRAIN
# -----------------------------
q1 = f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS duplicate_rows
FROM read_parquet('{FACT}')
WHERE report_date >= DATE '2026-03-01'
  AND report_date <  DATE '2026-04-01'
GROUP BY 1, 2, 3
HAVING COUNT(*) > 1
LIMIT 5
"""
grain_check = con.sql(q1).df()

print("QUERY 1 — grain duplicate probe")
display(grain_check)
print("Duplicate keys returned:", len(grain_check))
assert len(grain_check) == 0, "The stated daily grain does not hold for the March slice."


# ------------------------------------------
# Verification query 2 — COUNT + DATE SPAN
# ------------------------------------------
q2 = f"""
SELECT
    COUNT(*) AS march_rows,
    MIN(report_date) AS min_report_date,
    MAX(report_date) AS max_report_date
FROM read_parquet('{FACT}')
WHERE report_date >= DATE '2026-03-01'
  AND report_date <  DATE '2026-04-01'
"""
slice_check = con.sql(q2).df()

print("\nQUERY 2 — March slice count and date span")
display(slice_check)


# --------------------------------
# Verification query 3 — AVAILABILITY
# --------------------------------
q3 = f"""
SELECT
    COUNT(*) AS march_available_rows
FROM read_parquet('{FACT}')
WHERE report_date >= DATE '2026-03-01'
  AND report_date <  DATE '2026-04-01'
  AND gsc_data_available IS TRUE
"""
availability_check = con.sql(q3).df()

print("\nQUERY 3 — March GSC availability (IS TRUE)")
display(availability_check)


# ==========================================================
# Feature-building query — NOT one of the three verification
# queries above. It uses March for features and April only for
# the future development label.
# ==========================================================
feature_sql = f"""
WITH march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS march_impressions,
        SUM(gsc_clicks) AS march_clicks,
        SUM(gsc_sum_position) / NULLIF(SUM(gsc_impressions), 0) AS march_avg_position,
        COUNT(DISTINCT report_date) FILTER (WHERE gsc_impressions > 0) AS march_impression_days
    FROM read_parquet('{FACT}')
    WHERE report_date >= DATE '2026-03-01'
      AND report_date <  DATE '2026-04-01'
      AND gsc_data_available IS TRUE
    GROUP BY 1, 2
    HAVING SUM(gsc_impressions) >= 100
),
april AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS april_impressions,
        SUM(gsc_clicks) AS april_clicks,
        SUM(gsc_sum_position) / NULLIF(SUM(gsc_impressions), 0) AS april_avg_position
    FROM read_parquet('{FACT}')
    WHERE report_date >= DATE '2026-04-01'
      AND report_date <  DATE '2026-05-01'
      AND gsc_data_available IS TRUE
    GROUP BY 1, 2
    HAVING SUM(gsc_impressions) >= 100
),
april_with_tier AS (
    SELECT
        *,
        CASE
            WHEN april_avg_position <= 3 THEN 'top_3'
            WHEN april_avg_position <= 10 THEN 'page_1'
            WHEN april_avg_position <= 20 THEN 'striking'
            WHEN april_avg_position <= 50 THEN 'page_3_5'
            ELSE 'deep'
        END AS april_position_tier
    FROM april
    WHERE april_avg_position > 0
),
tier_baseline AS (
    SELECT
        april_position_tier,
        SUM(april_clicks) / NULLIF(SUM(april_impressions), 0) * 100 AS expected_april_ctr
    FROM april_with_tier
    GROUP BY 1
),
labeled AS (
    SELECT
        a.client_hash_id,
        a.content_hash_id,
        a.april_impressions,
        a.april_clicks,
        a.april_avg_position,
        a.april_position_tier,
        a.april_clicks / NULLIF(a.april_impressions, 0) * 100 AS april_ctr,
        b.expected_april_ctr,
        CASE
            WHEN a.april_clicks / NULLIF(a.april_impressions, 0) * 100 < b.expected_april_ctr
            THEN 1 ELSE 0
        END AS future_under_capture
    FROM april_with_tier a
    JOIN tier_baseline b USING (april_position_tier)
)
SELECT
    m.client_hash_id,
    m.content_hash_id,
    m.march_impressions,
    m.march_clicks,
    m.march_clicks / NULLIF(m.march_impressions, 0) * 100 AS march_ctr,
    m.march_avg_position,
    m.march_impression_days,
    l.future_under_capture
FROM march m
JOIN labeled l
  USING (client_hash_id, content_hash_id)
"""

frame = con.sql(feature_sql).df()

feature_cols = [
    "march_impressions",
    "march_clicks",
    "march_ctr",
    "march_avg_position",
    "march_impression_days",
]

print("\nFive-feature frame")
print("Rows:", f"{len(frame):,}")
display(frame[["client_hash_id", "content_hash_id"] + feature_cols + ["future_under_capture"]].head(10))

assert len(feature_cols) == 5
assert not any(c.startswith("april_") for c in feature_cols)


# -------------------------
# DELIBERATE LEAK EXPERIMENT
# -------------------------
# We intentionally add the future label itself as a feature.
# This is the trap: the model is allowed to see the answer.
model_df = frame.dropna(subset=feature_cols + ["future_under_capture"]).copy()
X_train, X_test, y_train, y_test = train_test_split(
    model_df[feature_cols],
    model_df["future_under_capture"],
    test_size=0.25,
    random_state=42,
    stratify=model_df["future_under_capture"],
)

honest_model = DecisionTreeClassifier(max_depth=3, random_state=42)
honest_model.fit(X_train, y_train)
honest_score = roc_auc_score(y_test, honest_model.predict_proba(X_test)[:, 1])

leaky_features = feature_cols + ["future_under_capture"]
X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(
    model_df[leaky_features],
    model_df["future_under_capture"],
    test_size=0.25,
    random_state=42,
    stratify=model_df["future_under_capture"],
)

leaky_model = DecisionTreeClassifier(max_depth=3, random_state=42)
leaky_model.fit(X_train_l, y_train_l)
leaky_score = roc_auc_score(y_test_l, leaky_model.predict_proba(X_test_l)[:, 1])

print("\nLeakage experiment")
print(f"Honest ROC-AUC: {honest_score:.3f}")
print(f"Leaky ROC-AUC:  {leaky_score:.3f}")
print("The leaky score should jump toward 1.0 because the model was given the answer.")

# Remove the leaked column and keep the honest feature set.
final_features = feature_cols.copy()
frame_honest = frame[["client_hash_id", "content_hash_id"] + final_features + ["future_under_capture"]].copy()

assert "future_under_capture" not in final_features
print("\nLeak removed.")
print("Final model features:", final_features)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

QUERY 1 — grain duplicate probe


,report_date,client_hash_id,content_hash_id,duplicate_rows


Duplicate keys returned: 0

QUERY 2 — March slice count and date span


,march_rows,min_report_date,max_report_date
0,9841378,2026-03-01,2026-03-31


...

,march_available_rows
0,3611061


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Five-feature frame
Rows: 88,482


,client_hash_id,content_hash_id,march_impressions,march_clicks,march_ctr,march_avg_position,march_impression_days,future_under_capture
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,1140.0,2.0,0.175439,4.450877,31,1
1,client_73cda7b4e4f265ea,content_05434271b257bb68,1421.0,6.0,0.422238,6.906404,31,0
2,client_73cda7b4e4f265ea,content_d056587ff7faca0c,2770.0,16.0,0.577617,3.950542,31,1
3,client_73cda7b4e4f265ea,content_2662845f598544ef,150.0,1.0,0.666667,7.506667,30,1
4,client_73cda7b4e4f265ea,content_712c365258cee05c,6048.0,23.0,0.380291,4.931878,31,1
5,client_73cda7b4e4f265ea,content_476c37c366920c1b,223.0,0.0,0.000000,61.538117,30,1
6,client_73cda7b4e4f265ea,content_098eedbd77ec1de1,281.0,1.0,0.355872,38.199288,30,1
7,client_73cda7b4e4f265ea,content_8935ed68eca88b01,4788.0,10.0,0.208855,9.556391,31,1
8,client_73cda7b4e4f265ea,content_91ffe8aef1f8c426,212.0,0.0,0.000000,6.698113,31,1
9,client_73cda7b4e4f265ea,content_5a87e33f97f030ec,178.0,0.0,0.000000,8.455056,31,1



Leakage experiment
Honest ROC-AUC: 0.772
Leaky ROC-AUC:  1.000
The leaky score should jump toward 1.0 because the model was given the answer.

Leak removed.
Final model features: ['march_impressions', 'march_clicks', 'march_ctr', 'march_avg_position', 'march_impression_days']


## 4. Data limits

**Named limitation:** the warehouse is an **unbalanced panel**. Clients have different history depths, so a calendar month is not equally observable for every client.

For this Lane 4 slice, I also deliberately restrict the development label to pages with measured GSC data and an April impression floor. That makes the outcome more interpretable, but it means the result is not a statement about pages with no measurable search data.

The March→April setup is a development window only. The final June 2026 month is treated as a sealed outcome/test month, not a month for developing the label logic.

Finally, a below-benchmark CTR is an **observed screening signal**, not proof that a title/meta/content change will cause an improvement. The score is decision-support for review prioritization.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.